# Forecasting Daily Minimum Electricity Demand in NSW for Grid Stability
**ENGG2112 Multidisciplinary Engineering Project: AI for Good**

### The problem
Rooftop solar now supplies a large share of NSW homes and businesses directly in the middle of the day. Less power is drawn from the grid, so **grid demand falls to very low levels around midday**, especially on sunny, mild spring days and public holidays. The Australian Energy Market Operator (AEMO) has identified minimum system load as a growing risk to grid stability. When demand is very low, too few large synchronous generators stay online to keep voltage and system strength stable, and operators may have to take emergency action such as curtailing rooftop solar.

Knowing **a day in advance** when demand will be dangerously low gives operators time to prepare. Accurate minimum-load forecasting therefore supports a grid that can safely absorb more renewable energy.

### What this notebook does
For each day, it forecasts two versions of the minimum load and produces two outputs:

| | Output 1: how low? (regression) | Output 2: is it risky? (classification) |
|---|---|---|
| **Daily minimum** (lowest half-hour of the whole day) | Forecast minimum in MW | Yes/no: is the day in the lowest 10% of days? |
| **Midday minimum** (lowest half-hour 10:00 to 15:00, the solar window) | Forecast minimum in MW | Yes/no flag, same rule |

Comparing the two targets shows how much of the daily low is now caused by solar at midday rather than by low overnight use.

### Forecasting setup (important for the report)
- **Forecast made at 00:00 for the day ahead.** Every input must be known by midnight: demand from yesterday or earlier, the calendar, and weather.
- **Weather assumption:** the dataset has *observed* temperature and wind, used here as a stand-in for a perfect weather forecast. A real system would use forecasts, so real-world error would be higher.
- **Seasonality is central.** Minimum load depends strongly on the time of year (sun angle, day length, heating and cooling). Section 6 builds features specifically for this, and Section 11 checks performance season by season.
- **Hold-out validation:** the last 3 months (**1 April to 30 June 2026**) are kept aside and used once at the end.

### How to run
Open in Google Colab and click **Runtime → Run all**. The normal CPU runtime is enough; the notebook takes about 1 to 2 minutes. Figures and tables are saved to a `results/` folder.

### Contents
1. Setup and configuration
2. Data collection
3. Data validation
4. From half-hourly data to one row per day
5. Exploratory analysis: seasonality of minimum load
6. Feature engineering
7. Train / hold-out split
8. Cross-validation setup
9. Regression baselines
10. Regression models and tuning
11. Seasonal performance check
12. Classification: flagging risky low-load days
13. Final evaluation on the hold-out period
14. Feature importance
15. Summary, limitations and ethics

## 1. Setup and configuration
**What this section does:** imports the libraries used in the notebook and sets every adjustable value in one place: the random seed, file location, hold-out start date, the midday window, and the threshold that defines a "risky" low-load day.

**Why:** keeping settings together makes the experiment reproducible (the fixed seed means every team member gets the same numbers) and easy to change. For example, to test a different midday window, change it here once.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import BaseEstimator, RegressorMixin, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              HistGradientBoostingClassifier)
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error, mean_absolute_percentage_error,
                             r2_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ---- Configuration ----
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_URL   = "https://raw.githubusercontent.com/TomSiede/NSW-demand-forecasting/main/data/NSW_ML_Training_Data_2024_2026.xlsx"
LOCAL_FILE = "NSW_ML_Training_Data_2024_2026.xlsx"   # used if the GitHub link fails (upload the file to Colab)
SHEET      = "ML Training Data"

HOLDOUT_START = "2026-04-01"     # last 3 months (Apr, May, Jun 2026) are held out
MIDDAY_SLOTS  = range(20, 30)    # half-hour slots starting 10:00 to 14:30, i.e. 10:00-15:00
RISK_QUANTILE = 0.10             # a "risky" day is in the lowest 10% of training days
N_FOLDS       = 5                # cross-validation folds (as in Tutorial 5)
SYDNEY_LAT    = -33.95           # latitude of Sydney Airport, used for day length

# The two minimum-load targets compared throughout the notebook
TARGETS = {"day_min": "Daily minimum (any time)",
           "mid_min": "Midday minimum (10:00-15:00)"}

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def save_fig(name):
    """Save the current figure into results/ for use in the report."""
    plt.savefig(os.path.join(RESULTS_DIR, name), dpi=150, bbox_inches="tight")

print(f"pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

## 2. Data collection
**What this section does:** loads the `ML Training Data` sheet from the team's GitHub repository. If that link fails (for example, the repository is private), it loads a copy uploaded to the Colab session instead. The rows are then sorted into time order.

**Why:** loading from GitHub means everyone uses the same version of the data. Sorting matters because daily summaries and lag features (Sections 4 and 6) assume the rows are in order.

**Sources** (from the dataset's Read Me sheet): AEMO half-hourly total demand for NSW1, Bureau of Meteorology temperature at Sydney Airport, three regional wind stations, and NSW public holidays from nsw.gov.au. Coverage: 1 July 2024 to 30 June 2026.

In [ ]:
try:
    raw = pd.read_excel(DATA_URL, sheet_name=SHEET)
    print("Loaded data from GitHub.")
except Exception as e:
    print(f"GitHub link failed ({type(e).__name__}); loading local file '{LOCAL_FILE}' instead.")
    raw = pd.read_excel(LOCAL_FILE, sheet_name=SHEET)

raw = raw.sort_values("Interval_Start_AEST").reset_index(drop=True)
print(f"{len(raw):,} half-hour rows x {raw.shape[1]} columns")
raw.head(3)

## 3. Data validation
**What this section does:** checks that there are no duplicate or missing timestamps, that every interval is exactly 30 minutes apart, that every day has all 48 half-hours, and that demand is never missing.

**Why:** a missing half-hour could make a day's minimum look higher than it really was, and gaps would shift the lag features. The notebook stops with an error rather than silently producing misleading results.

In [ ]:
ts = raw["Interval_Start_AEST"]
checks = {
    "No duplicate timestamps":       ts.duplicated().sum() == 0,
    "Every interval is 30 minutes":  (ts.diff().dropna() == pd.Timedelta(minutes=30)).all(),
    "Every day has 48 half-hours":   (raw.groupby(ts.dt.normalize()).size() == 48).all(),
    "No missing demand values":      raw["Demand_MW"].isna().sum() == 0,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), "Data validation failed: check the dataset before continuing."

missing = raw.isna().sum()
print("\nMissing weather values (handled later, inside the model pipeline):")
print(missing[(missing > 0) & (missing.index != "NSW_Holiday_Name")].to_string())

## 4. From half-hourly data to one row per day
**What this section does:** the question is now "how low will demand get *today*?", so the 35,040 half-hourly rows are summarised into **730 daily rows**. For each day it calculates:
- **`day_min`:** the lowest half-hour demand of the whole day (target 1).
- **`mid_min`:** the lowest half-hour demand between 10:00 and 15:00 (target 2).
- **`min_hour`:** the time the daily minimum occurred, used to show where the low point falls.
- **Daily weather:** mean, maximum and minimum temperature, mean midday temperature, and mean wind speed.
- **Day type:** weekend and public holiday flags.

The electricity price column is not used, because price is set at the same time as demand and would leak information (see the Read Me sheet).

**Why:** this changes the unit of prediction from a half-hour to a day, which matches how an operator would plan: "Tomorrow's minimum will be about X MW." Fewer rows also means much faster training.

In [ ]:
hh = raw.copy()
hh["Date"] = hh["Interval_Start_AEST"].dt.normalize()
hh["Is_Midday"] = hh["TOD_Slot_0_47"].isin(MIDDAY_SLOTS)
by_day = hh.groupby("Date")

daily = pd.DataFrame({
    # ---- targets ----
    "day_min":  by_day["Demand_MW"].min(),
    "mid_min":  hh[hh["Is_Midday"]].groupby("Date")["Demand_MW"].min(),
    # hour of day when the daily minimum happened (slot / 2 = hour)
    "min_hour": hh.loc[by_day["Demand_MW"].idxmin(), ["Date", "TOD_Slot_0_47"]]
                  .set_index("Date")["TOD_Slot_0_47"] / 2,
    "day_mean": by_day["Demand_MW"].mean(),           # used only to build a lag feature
    # ---- daily weather (observed, standing in for a forecast) ----
    "Temp_mean":   by_day["Sydney_Airport_Temp_C"].mean(),
    "Temp_max":    by_day["Sydney_Airport_Temp_C"].max(),
    "Temp_min":    by_day["Sydney_Airport_Temp_C"].min(),
    "Temp_midday": hh[hh["Is_Midday"]].groupby("Date")["Sydney_Airport_Temp_C"].mean(),
    "Wind_mean":   by_day["Wind_NSW_Weighted_mps"].mean(),
    # ---- calendar ----
    "DayOfWeek":   by_day["DayOfWeek_Mon0"].first(),
    "Is_Weekend":  by_day["Is_Weekend"].first(),
    "Is_Holiday":  by_day["Is_NSW_Public_Holiday"].max(),
})
daily.index.name = "Date"

midday_share = daily["min_hour"].between(10, 14.5).mean()
print(f"{len(daily)} days | the daily minimum fell in the 10:00-15:00 window on {midday_share:.0%} of days")
daily.head()

## 5. Exploratory analysis: seasonality of minimum load
**What this section does:** plots how minimum load changes through the year and what drives it:
- **(a)** both targets day by day across the two years, with the 10% risk threshold;
- **(b)** the monthly pattern of each target;
- **(c)** the time of day when the minimum occurs, by month;
- **(d)** minimum load against day length (a proxy for how much solar energy is available).

**Why:** this justifies the modelling choices. If minimum load follows a strong yearly cycle, the model needs seasonal features (Section 6) and must be checked across all seasons (Section 11), not just the hold-out months.

**What to look for:** the lowest days cluster in **spring (September to November)** and the **Christmas to New Year** break, when the sun is strong, temperatures are mild (little heating or air conditioning) and many businesses are closed. Winter minimums are much higher because of heating and short days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# (a) both targets over time, with the risk threshold computed from pre-hold-out days
pre = daily[daily.index < HOLDOUT_START]
ax = axes[0, 0]
for col, colour in [("day_min", "#1f77b4"), ("mid_min", "#ff7f0e")]:
    ax.plot(daily.index, daily[col], lw=0.7, color=colour, label=TARGETS[col])
ax.axhline(pre["day_min"].quantile(RISK_QUANTILE), color="red", ls="--", lw=1, label="Lowest-10% threshold")
ax.axvspan(pd.Timestamp(HOLDOUT_START), daily.index.max(), color="grey", alpha=0.15, label="Hold-out period")
ax.set(title="(a) Daily minimum demand, Jul 2024 to Jun 2026", ylabel="MW"); ax.legend(fontsize=8)

# (b) monthly pattern
ax = axes[0, 1]
monthly = daily.groupby(daily.index.month)[["day_min", "mid_min"]].mean()
ax.plot(monthly.index, monthly["day_min"], marker="o", label=TARGETS["day_min"])
ax.plot(monthly.index, monthly["mid_min"], marker="s", label=TARGETS["mid_min"])
ax.set(title="(b) Average minimum by month", xlabel="Month", ylabel="MW", xticks=range(1, 13)); ax.legend(fontsize=8)

# (c) when does the minimum happen?
ax = axes[1, 0]
ax.scatter(daily.index.month + np.random.uniform(-0.3, 0.3, len(daily)), daily["min_hour"], s=6, alpha=0.5)
ax.axhspan(10, 15, color="gold", alpha=0.2, label="Midday window")
ax.set(title="(c) Time of day of the daily minimum", xlabel="Month", ylabel="Hour", xticks=range(1, 13),
       yticks=range(0, 25, 3)); ax.legend(fontsize=8)

# (d) day length vs minimum (day length formula explained in Section 6)
doy = daily.index.dayofyear
decl = np.radians(23.44) * np.sin(2 * np.pi * (284 + doy) / 365)
day_len = 24 / np.pi * np.arccos(-np.tan(np.radians(SYDNEY_LAT)) * np.tan(decl))
ax = axes[1, 1]
sc = ax.scatter(day_len, daily["mid_min"], c=daily["Temp_mean"], cmap="coolwarm", s=8)
plt.colorbar(sc, ax=ax, label="Mean temperature (°C)")
ax.set(title="(d) Midday minimum vs day length", xlabel="Hours of daylight", ylabel="MW")

plt.tight_layout(); save_fig("01_seasonality.png"); plt.show()

print("Lowest 8 days in the dataset:")
print(daily.nsmallest(8, "day_min")[["day_min", "min_hour", "Temp_mean", "Is_Weekend", "Is_Holiday"]].round(1))

## 6. Feature engineering
**What this section does:** creates the inputs the models learn from, with particular attention to **seasonality**. Every feature is known by midnight at the start of the forecast day.

| Group | Features | Why they help |
|---|---|---|
| **Season (smooth)** | `DOY_sin1`, `DOY_cos1`, `DOY_sin2`, `DOY_cos2` | Day of year encoded as sine and cosine waves, so 31 December sits next to 1 January. The second pair (twice a year) captures the two seasonal lows seen in Section 5, spring and late summer. |
| **Season (physical)** | `Day_Length_h` | Hours of daylight in Sydney, calculated from astronomy. A direct proxy for how much rooftop solar the season allows. |
| **Season (category)** | `Month` | Lets tree models learn month-specific effects. |
| **Calendar** | `DayOfWeek`, `Is_Weekend`, `Is_Holiday`, `Prev_Holiday`, `Next_Holiday`, `Xmas_Break` | Weekends and holidays have lower business demand. The Christmas to New Year break produces several of the lowest days in the data. |
| **Weather** | `Temp_mean`, `Temp_max`, `Temp_min`, `Temp_midday`, `HDD`, `CDD`, `Wind_mean` | Mild days need little heating or cooling, so demand drops. Heating degrees (`HDD`, below 18 °C) and cooling degrees (`CDD`, above 22 °C) capture this U-shaped effect. |
| **Recent minimums** | `*_lag1`, `*_lag7`, `*_roll7` for both targets, plus `day_mean_lag1` | Yesterday's minimum, the same weekday last week, and the average over the past 7 days. These track the current seasonal level and recent trends such as solar growth. |

**Why:** in a dataset with only two years, the model sees each season only once or twice. Smooth seasonal features (sine/cosine and day length) let it generalise between nearby dates instead of memorising specific months.

**Missing:** the dataset has no cloud cover or solar irradiance, which is the biggest driver of midday minimum load. Temperature only partly stands in for it (see Section 15).

In [ ]:
HDD_BASE, CDD_BASE = 18.0, 22.0

def build_features(d):
    """Create all model inputs for each day. Uses only information known at 00:00 on that day."""
    d = d.copy()
    doy = d.index.dayofyear

    # --- Seasonality: smooth yearly (1x) and half-yearly (2x) cycles ---
    for k in (1, 2):
        d[f"DOY_sin{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
        d[f"DOY_cos{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    d["Month"] = d.index.month

    # --- Seasonality: hours of daylight in Sydney (standard solar declination formula) ---
    declination = np.radians(23.44) * np.sin(2 * np.pi * (284 + doy) / 365)
    d["Day_Length_h"] = 24 / np.pi * np.arccos(-np.tan(np.radians(SYDNEY_LAT)) * np.tan(declination))

    # --- Calendar: holidays are known in advance, so yesterday's and tomorrow's are both allowed ---
    d["Prev_Holiday"] = d["Is_Holiday"].shift(1).fillna(0).astype(int)
    d["Next_Holiday"] = d["Is_Holiday"].shift(-1).fillna(0).astype(int)
    d["Xmas_Break"] = (((d.index.month == 12) & (d.index.day >= 24)) |
                       ((d.index.month == 1) & (d.index.day <= 2))).astype(int)

    # --- Weather: heating and cooling degrees from daily mean temperature ---
    d["HDD"] = np.maximum(HDD_BASE - d["Temp_mean"], 0)
    d["CDD"] = np.maximum(d["Temp_mean"] - CDD_BASE, 0)

    # --- Recent minimums: shift(1) = yesterday, so nothing from the forecast day itself is used ---
    for t in TARGETS:
        d[f"{t}_lag1"] = d[t].shift(1)
        d[f"{t}_lag7"] = d[t].shift(7)
        d[f"{t}_roll7"] = d[t].shift(1).rolling(7).mean()
    d["day_mean_lag1"] = d["day_mean"].shift(1)
    return d

FEATURES = [
    "DOY_sin1", "DOY_cos1", "DOY_sin2", "DOY_cos2", "Day_Length_h", "Month",
    "DayOfWeek", "Is_Weekend", "Is_Holiday", "Prev_Holiday", "Next_Holiday", "Xmas_Break",
    "Temp_mean", "Temp_max", "Temp_min", "Temp_midday", "HDD", "CDD", "Wind_mean",
    "day_min_lag1", "day_min_lag7", "day_min_roll7",
    "mid_min_lag1", "mid_min_lag7", "mid_min_roll7", "day_mean_lag1",
]

feat = build_features(daily)
# The first 7 days have no "last week" value, so they are removed
feat = feat.dropna(subset=["day_min_lag7", "mid_min_lag7", "day_min_roll7"])
print(f"{len(FEATURES)} features | {len(feat)} usable days (first week removed)")
feat[FEATURES].describe().T[["mean", "min", "max"]].round(2)

## 7. Train / hold-out split
**What this section does:** splits the days by date. Everything before **1 April 2026** is used for training and cross-validation. **April to June 2026** (the last 3 months) is the hold-out validation set, which stays untouched until Section 13.

It also sets the **risk threshold** for the yes/no flag: a day is "risky" if its minimum is in the lowest 10% of *training* days. The threshold comes from training data only, so the hold-out period cannot influence it.

**Why a date split:** a random split would let the model learn from days after the ones it predicts. Splitting by date mirrors real use: learn from the past, forecast the future.

**Key finding:** the two targets produce the **same threshold and the same risky days**. Every one of the lowest 10% of days had its minimum in the midday solar window, so on the days that matter for grid stability, the daily minimum *is* the midday minimum. The yes/no results in Sections 12 and 13 are therefore identical for both targets. This is not a bug; it is evidence that rooftop solar now drives the riskiest low-load events.

**Seasonal caution:** the hold-out months are autumn and early winter, which have higher minimums and few risky days. The hold-out result therefore shows performance for that season only. Section 11 uses cross-validation, which covers all seasons, to fill this gap.

In [ ]:
train = feat[feat.index < HOLDOUT_START]
hold  = feat[feat.index >= HOLDOUT_START]
X_train, X_hold = train[FEATURES], hold[FEATURES]

# Risk threshold for each target, from training days only
THRESHOLDS = {t: train[t].quantile(RISK_QUANTILE) for t in TARGETS}

print(f"Training: {len(train)} days ({train.index.min():%d %b %Y} to {train.index.max():%d %b %Y})")
print(f"Hold-out: {len(hold)} days ({hold.index.min():%d %b %Y} to {hold.index.max():%d %b %Y})\n")
for t, name in TARGETS.items():
    n_tr = (train[t] < THRESHOLDS[t]).sum()
    n_ho = (hold[t] < THRESHOLDS[t]).sum()
    print(f"{name}: risky if below {THRESHOLDS[t]:,.0f} MW | risky days: {n_tr} in training, {n_ho} in hold-out")

same = ((train["day_min"] < THRESHOLDS["day_min"]) == (train["mid_min"] < THRESHOLDS["mid_min"])).all()
print(f"\nThe two targets flag exactly the same risky days: {same}")

## 8. Cross-validation setup
**What this section does:** sets up five-fold **time-series** cross-validation and a helper, `cv_predict`, that collects each model's predictions on every validation fold.

How the folds work: the training days are cut into consecutive blocks of about 100 days. Each fold trains on all earlier days and predicts the next block. Because the blocks march forward through time, **the five validation blocks together cover every season**, from late spring 2024 to summer 2026.

**Why:** Tutorial 5 introduced five-fold cross-validation so models can be compared without touching the hold-out set. Ordinary k-fold shuffles days, which would let the model see the future. Pooling the predictions from all five folds also lets Section 11 check accuracy month by month, and gives enough risky days to judge the yes/no flag fairly.

**Metrics used:**
- Regression: **MAE** (average error in MW), **RMSE** (punishes large misses more), **MAPE** (% error) and **R²** (share of variation explained).
- Classification: **Recall** (share of truly risky days that were flagged; the most important, since a missed risky day is the dangerous mistake), **Precision** (share of flags that were correct; low precision means false alarms), **F1** (balance of the two) and **ROC-AUC** (how well the model ranks risky days above safe days, from 0.5 = random to 1 = perfect).

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

for i, (tr, va) in enumerate(tscv.split(X_train), 1):
    print(f"Fold {i}: train {X_train.index[tr[0]]:%d %b %Y} to {X_train.index[tr[-1]]:%d %b %Y} "
          f"({len(tr)} days) -> validate {X_train.index[va[0]]:%d %b %Y} to {X_train.index[va[-1]]:%d %b %Y}")

def cv_predict(model, y, proba=False):
    """Fit a fresh copy of the model on each fold's past and predict its validation block.
    Returns a Series of out-of-fold predictions (NaN for days never used for validation)
    and the fold number of each prediction."""
    pred = pd.Series(np.nan, index=y.index)
    fold = pd.Series(np.nan, index=y.index)
    for k, (tr, va) in enumerate(tscv.split(X_train), 1):
        m = clone(model).fit(X_train.iloc[tr], y.iloc[tr])
        pred.iloc[va] = m.predict_proba(X_train.iloc[va])[:, 1] if proba else m.predict(X_train.iloc[va])
        fold.iloc[va] = k
    return pred, fold

def regression_scores(y_true, y_pred):
    return {"MAE (MW)": mean_absolute_error(y_true, y_pred),
            "RMSE (MW)": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE (%)": mean_absolute_percentage_error(y_true, y_pred) * 100,
            "R2": r2_score(y_true, y_pred)}

## 9. Regression baselines
**What this section does:** measures three simple forecasting rules for each target:
1. **Same as yesterday:** tomorrow's minimum equals today's.
2. **Same weekday last week:** uses the minimum from 7 days earlier.
3. **Seasonal average:** the average minimum in the training data for that month and day type (weekday, weekend or holiday). This is a pure seasonality rule.

**Why:** an ML model is only useful if it clearly beats simple rules. The seasonal-average baseline is especially important here: if the ML model cannot beat it, the model is learning nothing beyond the calendar.

In [ ]:
class ColumnBaseline(BaseEstimator, RegressorMixin):
    """Predicts by copying one input column (e.g. yesterday's minimum). Nothing is learned."""
    def __init__(self, column="day_min_lag1"):
        self.column = column
    def fit(self, X, y):
        return self
    def predict(self, X):
        return X[self.column].to_numpy()

class SeasonalAverage(BaseEstimator, RegressorMixin):
    """Predicts the average training value for each (month, day type) combination."""
    @staticmethod
    def _key(X):
        day_type = np.where(X["Is_Holiday"] == 1, 2, X["Is_Weekend"])   # 0 weekday, 1 weekend, 2 holiday
        return pd.MultiIndex.from_arrays([X["Month"].to_numpy(), day_type])
    def fit(self, X, y):
        s = pd.Series(np.asarray(y), index=self._key(X))
        self.table_ = s.groupby(level=[0, 1]).mean()
        self.month_ = s.groupby(level=0).mean()      # fallback if a combination was never seen
        self.overall_ = s.mean()
        return self
    def predict(self, X):
        key = self._key(X)
        out = self.table_.reindex(key).to_numpy()
        fallback = self.month_.reindex(key.get_level_values(0)).fillna(self.overall_).to_numpy()
        return np.where(np.isnan(out), fallback, out)

def make_baselines(t):
    return {"Baseline: same as yesterday":   ColumnBaseline(f"{t}_lag1"),
            "Baseline: same day last week":  ColumnBaseline(f"{t}_lag7"),
            "Baseline: seasonal average":    SeasonalAverage()}

cv_rows, oof = [], {}       # oof[(target, model)] stores out-of-fold predictions for later sections
for t, tname in TARGETS.items():
    y = train[t]
    for name, model in make_baselines(t).items():
        pred, fold = cv_predict(model, y)
        mask = pred.notna()
        oof[(t, name)] = pred
        cv_rows.append({"Target": tname, "Model": name, **regression_scores(y[mask], pred[mask])})

pd.DataFrame(cv_rows).round(2)

## 10. Regression models and tuning
**What this section does:** trains three machine learning models for each target and compares them with the baselines on the same folds, then tunes the Gradient Boosting model.

| Model | How it works |
|---|---|
| **Linear Regression (Ridge)** | A weighted sum of the features, with a penalty that keeps the weights small. Simple and interpretable; features are scaled first. |
| **Random Forest** | Averages many decision trees, each trained on a random sample of days (the decision trees from lectures, combined). |
| **Gradient Boosting** | Builds trees one after another, each correcting the earlier trees' errors. Handles missing values on its own. |

Tuning searches combinations of `learning_rate` (how big each correction is), `max_iter` (number of trees), `max_leaf_nodes` (tree complexity) and `min_samples_leaf` (minimum days per leaf, which limits overfitting) using the same time-series folds.

**Why Ridge instead of plain linear regression:** plain linear regression failed badly in testing (errors over 15,000 MW). The first cross-validation fold trains on only July to October, so several seasonal features move almost in lockstep, and the model assigned them huge opposite weights that exploded on later dates. Ridge's penalty prevents this. This is a good "Issues Faced" example for the report.

**Why no neural network this time:** with only about 630 training days, a neural network has too little data to learn reliably and would mostly overfit. Tree models are the standard choice for small tabular datasets. This is a good point to justify in the report.

**Why:** models are wrapped in a `Pipeline` so that filling missing values and scaling are re-learned inside each fold, preventing leakage from validation days.

In [ ]:
def make_models():
    return {
        # Ridge = linear regression with a penalty on large weights (see the note above)
        "Linear Regression (Ridge)": Pipeline([("impute", SimpleImputer(strategy="median")),
                                               ("scale", StandardScaler()),
                                               ("model", Ridge(alpha=10.0))]),
        "Random Forest": Pipeline([("impute", SimpleImputer(strategy="median")),
                                   ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                                                   n_jobs=-1, random_state=RANDOM_STATE))]),
        "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                           random_state=RANDOM_STATE),
    }

PARAM_GRID = {"learning_rate": [0.03, 0.1], "max_iter": [200, 500],
              "max_leaf_nodes": [7, 15], "min_samples_leaf": [10, 20]}

tuned = {}   # best tuned Gradient Boosting model for each target
for t, tname in TARGETS.items():
    y = train[t]
    for name, model in make_models().items():
        pred, fold = cv_predict(model, y)
        mask = pred.notna()
        oof[(t, name)] = pred
        cv_rows.append({"Target": tname, "Model": name, **regression_scores(y[mask], pred[mask])})

    # Tune Gradient Boosting with the same time-series folds
    search = GridSearchCV(HistGradientBoostingRegressor(random_state=RANDOM_STATE), PARAM_GRID,
                          cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_train, y)
    tuned[t] = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **search.best_params_)
    pred, fold = cv_predict(tuned[t], y)
    mask = pred.notna()
    oof[(t, "Gradient Boosting (tuned)")] = pred
    cv_rows.append({"Target": tname, "Model": "Gradient Boosting (tuned)", **regression_scores(y[mask], pred[mask])})
    print(f"{tname}: best settings {search.best_params_}")

cv_table = pd.DataFrame(cv_rows).sort_values(["Target", "MAE (MW)"]).reset_index(drop=True)
cv_table.to_csv(os.path.join(RESULTS_DIR, "cv_regression_results.csv"), index=False)
cv_table.round(2)

In [ ]:
# Chart: cross-validated MAE for every model, both targets side by side (lower is better)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharex=True)
for ax, (t, tname) in zip(axes, TARGETS.items()):
    sub = cv_table[cv_table["Target"] == tname].sort_values("MAE (MW)")
    colours = ["#bbbbbb" if m.startswith("Baseline") else "#4c9be8" for m in sub["Model"]]
    ax.barh(sub["Model"], sub["MAE (MW)"], color=colours)
    ax.invert_yaxis(); ax.set(title=tname, xlabel="Cross-validated MAE (MW)")
fig.suptitle("Model comparison (grey = baseline)")
plt.tight_layout(); save_fig("02_model_comparison.png"); plt.show()

# Pick the best ML model (lowest CV MAE) for each target
best_reg = {}
for t, tname in TARGETS.items():
    sub = cv_table[(cv_table["Target"] == tname) & ~cv_table["Model"].str.startswith("Baseline")]
    best_reg[t] = sub.iloc[0]["Model"]
    print(f"Best model for {tname}: {best_reg[t]}")

## 11. Seasonal performance check
**What this section does:** uses the pooled cross-validation predictions from Section 10, which span late 2024 to early 2026, to show the best model's error **for each season** and compares it with the seasonal-average baseline.

**Why:** a large part of the prediction is seasonal, and the hold-out months only cover autumn and winter. This section answers the questions the hold-out set cannot: does the model stay accurate in **spring**, when the riskiest low-load days happen? Does it consistently over- or under-predict in any season? A model that is accurate on average but poor in spring would be least reliable exactly when it matters most.

**Bias** = average of (predicted − actual). A positive bias in spring means the model predicts minimums that are *too high*, so it would underestimate how low demand will go, which is the dangerous direction for grid stability.

In [ ]:
SEASONS = {12: "Summer", 1: "Summer", 2: "Summer", 3: "Autumn", 4: "Autumn", 5: "Autumn",
           6: "Winter", 7: "Winter", 8: "Winter", 9: "Spring", 10: "Spring", 11: "Spring"}
season_order = ["Spring", "Summer", "Autumn", "Winter"]

season_rows = []
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (t, tname) in zip(axes, TARGETS.items()):
    y = train[t]
    for model_name in [best_reg[t], "Baseline: seasonal average"]:
        p = oof[(t, model_name)].dropna()
        err = (p - y[p.index])
        season = pd.Series(p.index.month.map(SEASONS), index=p.index)
        for s in season_order:
            e = err[season == s]
            season_rows.append({"Target": tname, "Model": model_name, "Season": s, "Days": len(e),
                                "MAE (MW)": e.abs().mean(), "Bias (MW)": e.mean()})
        # monthly MAE line for the chart
        ax.plot(range(1, 13), err.abs().groupby(p.index.month).mean().reindex(range(1, 13)),
                marker="o", label=model_name)
    ax.set(title=f"{tname}: cross-validated error by month", xlabel="Month", ylabel="MAE (MW)",
           xticks=range(1, 13))
    ax.legend(fontsize=8)
plt.tight_layout(); save_fig("03_error_by_month.png"); plt.show()

season_table = pd.DataFrame(season_rows)
season_table.to_csv(os.path.join(RESULTS_DIR, "seasonal_performance.csv"), index=False)
season_table.pivot_table(index=["Target", "Model"], columns="Season",
                         values="MAE (MW)")[season_order].round(0)

In [ ]:
# Bias by season for the best model (positive = predicted minimum too high = risk underestimated)
season_table[~season_table["Model"].str.startswith("Baseline")].pivot_table(
    index="Target", columns="Season", values="Bias (MW)")[season_order].round(0)

## 12. Classification: flagging risky low-load days
**What this section does:** answers the yes/no question "will tomorrow be a risky low-load day?" A day is risky if its minimum is below the threshold set in Section 7 (the lowest 10% of training days). Four approaches are compared using the same cross-validation folds:

| Approach | How it decides |
|---|---|
| **Baseline: yesterday was risky** | Flags a day if the previous day was below the threshold. |
| **Regression-based** | Uses the best regression model from Section 10 and flags the day if the *predicted* minimum is below the threshold. |
| **Logistic Regression** | A classifier that outputs the probability of a risky day. |
| **Gradient Boosting classifier** | A tree-based classifier, the classification version of the Section 10 model. |

Both classifiers use **class weighting** (`class_weight="balanced"`). Only about 1 day in 10 is risky, so without weighting a model could score 90% accuracy by *never* raising a flag, which would be useless. Weighting makes missing a risky day cost more during training. For the same reason, **accuracy is not reported**: recall and precision are the honest measures.

**Why:** operators need a clear action trigger as well as a number. The comparison also shows whether a dedicated classifier beats simply thresholding the regression forecast.

In [ ]:
class ThresholdBaseline(BaseEstimator, ClassifierMixin):
    """Flags a day as risky if yesterday's minimum was below the threshold."""
    def __init__(self, column="day_min_lag1", threshold=0.0):
        self.column, self.threshold = column, threshold
    def fit(self, X, y):
        self.classes_ = np.array([0, 1]); return self
    def predict(self, X):
        return (X[self.column] < self.threshold).astype(int).to_numpy()

def make_classifiers(t):
    return {
        "Logistic Regression": Pipeline([("impute", SimpleImputer(strategy="median")),
                                         ("scale", StandardScaler()),
                                         ("model", LogisticRegression(class_weight="balanced", max_iter=2000,
                                                                      C=0.5))]),
        "Gradient Boosting classifier": HistGradientBoostingClassifier(
            class_weight="balanced", max_iter=200, learning_rate=0.05, max_leaf_nodes=7,
            min_samples_leaf=10, random_state=RANDOM_STATE),
    }

def classification_scores(y_true, flag, prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, flag, labels=[0, 1]).ravel()
    return {"Risky days": int(y_true.sum()), "Caught": int(tp), "Missed": int(fn), "False alarms": int(fp),
            "Recall": recall_score(y_true, flag, zero_division=0),
            "Precision": precision_score(y_true, flag, zero_division=0),
            "F1": f1_score(y_true, flag, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, prob) if prob is not None and len(set(y_true)) == 2 else np.nan}

clf_rows, oof_flags = [], {}
for t, tname in TARGETS.items():
    thr = THRESHOLDS[t]
    y_cls = (train[t] < thr).astype(int)
    mask = oof[(t, best_reg[t])].notna()          # the days covered by the validation folds

    # Baseline
    base_flag, _ = cv_predict(ThresholdBaseline(f"{t}_lag1", thr), y_cls)
    clf_rows.append({"Target": tname, "Model": "Baseline: yesterday was risky",
                     **classification_scores(y_cls[mask], base_flag[mask].astype(int))})

    # Regression-based: flag when the predicted minimum is below the threshold.
    # A lower predicted value means more risk, so -prediction works as a risk score for ROC-AUC.
    reg_pred = oof[(t, best_reg[t])]
    reg_flag = (reg_pred < thr).astype(int)
    oof_flags[(t, "Regression-based")] = reg_flag[mask]
    clf_rows.append({"Target": tname, "Model": f"Regression-based ({best_reg[t]})",
                     **classification_scores(y_cls[mask], reg_flag[mask], -reg_pred[mask])})

    # Dedicated classifiers
    for name, model in make_classifiers(t).items():
        prob, _ = cv_predict(model, y_cls, proba=True)
        flag = (prob[mask] >= 0.5).astype(int)
        oof_flags[(t, name)] = flag
        clf_rows.append({"Target": tname, "Model": name, **classification_scores(y_cls[mask], flag, prob[mask])})

clf_table = pd.DataFrame(clf_rows).sort_values(["Target", "F1"], ascending=[True, False]).reset_index(drop=True)
clf_table.to_csv(os.path.join(RESULTS_DIR, "cv_classification_results.csv"), index=False)
clf_table.round(3)

In [ ]:
# Pick the best flagging approach per target by F1 (balances catching risky days against false alarms)
best_clf = {}
for t, tname in TARGETS.items():
    sub = clf_table[(clf_table["Target"] == tname) & ~clf_table["Model"].str.startswith("Baseline")]
    best_clf[t] = sub.iloc[0]["Model"]
    print(f"Best flagging approach for {tname}: {best_clf[t]} "
          f"(recall {sub.iloc[0]['Recall']:.2f}, precision {sub.iloc[0]['Precision']:.2f})")

# When during the year were risky days flagged and missed? (cross-validation period)
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for ax, (t, tname) in zip(axes, TARGETS.items()):
    key = "Regression-based" if best_clf[t].startswith("Regression") else best_clf[t]
    flag = oof_flags[(t, key)]
    actual = train.loc[flag.index, t]
    risky = actual < THRESHOLDS[t]
    ax.plot(actual.index, actual, lw=0.8, color="grey", label="Actual minimum")
    ax.axhline(THRESHOLDS[t], color="red", ls="--", lw=1, label="Risk threshold")
    ax.scatter(actual.index[risky & (flag == 1)], actual[risky & (flag == 1)], color="green", s=25, label="Caught")
    ax.scatter(actual.index[risky & (flag == 0)], actual[risky & (flag == 0)], color="red", marker="x", s=40, label="Missed")
    ax.scatter(actual.index[~risky & (flag == 1)], actual[~risky & (flag == 1)], color="orange", s=15, label="False alarm")
    ax.set(title=f"{tname}: {best_clf[t]} (cross-validation)", ylabel="MW"); ax.legend(fontsize=8, ncol=5)
plt.tight_layout(); save_fig("04_risk_flags_cv.png"); plt.show()

## 13. Final evaluation on the hold-out period (April to June 2026)
**What this section does:** retrains the selected models on **all** training days and evaluates them **once** on the 3-month hold-out period:
1. **Regression:** the best model for each target against the three baselines.
2. **Classification:** the best flagging approach for each target, with a confusion matrix.

**Why:** the hold-out months have not influenced any choice so far, so this is an honest test on unseen data. Do not go back and change the models based on these numbers; that would turn the hold-out set into another validation set and make the result over-optimistic.

**Reading the classification result:** April to June contains very few risky days (see Section 7), so recall and precision here rest on a handful of examples. The cross-validation results in Section 12 are the more reliable measure of flagging performance, and this section mainly checks that the model does not raise many false alarms in a low-risk season.

In [ ]:
hold_rows, hold_pred = [], {}
for t, tname in TARGETS.items():
    candidates = {**make_baselines(t), **make_models(), "Gradient Boosting (tuned)": tuned[t]}
    for name in list(make_baselines(t)) + [best_reg[t]]:
        m = clone(candidates[name]).fit(X_train, train[t])
        p = pd.Series(m.predict(X_hold), index=hold.index)
        if name == best_reg[t]:
            hold_pred[t] = p
        hold_rows.append({"Target": tname, "Model": name, **regression_scores(hold[t], p)})

hold_table = pd.DataFrame(hold_rows).sort_values(["Target", "MAE (MW)"]).reset_index(drop=True)
hold_table.to_csv(os.path.join(RESULTS_DIR, "holdout_regression_results.csv"), index=False)
hold_table.round(2)

In [ ]:
# Actual vs predicted minimum load over the hold-out period
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for ax, (t, tname) in zip(axes, TARGETS.items()):
    ax.plot(hold.index, hold[t], marker="o", ms=3, label="Actual")
    ax.plot(hold.index, hold_pred[t], marker="s", ms=3, alpha=0.8, label=f"Predicted ({best_reg[t]})")
    ax.axhline(THRESHOLDS[t], color="red", ls="--", lw=1, label="Risk threshold")
    mae = mean_absolute_error(hold[t], hold_pred[t])
    ax.set(title=f"{tname}: hold-out period (MAE {mae:,.0f} MW)", ylabel="MW"); ax.legend(fontsize=8)
plt.tight_layout(); save_fig("05_holdout_actual_vs_predicted.png"); plt.show()

In [ ]:
# Classification on the hold-out period
hold_clf_rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (t, tname) in zip(axes, TARGETS.items()):
    thr = THRESHOLDS[t]
    y_hold_cls = (hold[t] < thr).astype(int)
    if best_clf[t].startswith("Regression"):
        flag = (hold_pred[t] < thr).astype(int); score = -hold_pred[t]
    else:
        m = clone(make_classifiers(t)[best_clf[t]]).fit(X_train, (train[t] < thr).astype(int))
        score = pd.Series(m.predict_proba(X_hold)[:, 1], index=hold.index)
        flag = (score >= 0.5).astype(int)
    hold_clf_rows.append({"Target": tname, "Model": best_clf[t], **classification_scores(y_hold_cls, flag, score)})
    ConfusionMatrixDisplay(confusion_matrix(y_hold_cls, flag, labels=[0, 1]),
                           display_labels=["Normal", "Risky"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set(title=f"{tname}\n{best_clf[t]}"); ax.grid(False)
plt.tight_layout(); save_fig("06_holdout_confusion_matrices.png"); plt.show()

hold_clf_table = pd.DataFrame(hold_clf_rows)
hold_clf_table.to_csv(os.path.join(RESULTS_DIR, "holdout_classification_results.csv"), index=False)
hold_clf_table.round(3)

## 14. Feature importance
**What this section does:** measures how much each feature matters to the final regression model for each target. **Permutation importance** shuffles one feature at a time and records how much the error increases. A big increase means the model depends heavily on that feature. It is calculated on the training data, which covers every season (the hold-out months alone cover only autumn and winter).

**Why:** this makes the model more transparent (a key criterion from the Week 6 tutorial) and shows how much of the forecast comes from **seasonality** (the yellow bars) compared with recent demand and weather. If the model relied on something physically meaningless, this is where it would show up.

**Interpreting the seasonal share:** the explicit seasonal features may look less important than expected. That is because the recent-minimum features (`*_lag1`, `*_roll7`) already carry the seasonal level: last week's minimum in October is already low. Seasonality is still driving the forecast, mostly *through* those features.

In [ ]:
SEASONAL_FEATURES = {"DOY_sin1", "DOY_cos1", "DOY_sin2", "DOY_cos2", "Day_Length_h", "Month"}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, (t, tname) in zip(axes, TARGETS.items()):
    candidates = {**make_models(), "Gradient Boosting (tuned)": tuned[t]}
    m = clone(candidates[best_reg[t]]).fit(X_train, train[t])
    imp = permutation_importance(m, X_train, train[t], n_repeats=10, scoring="neg_mean_absolute_error",
                                 random_state=RANDOM_STATE, n_jobs=-1)
    s = pd.Series(imp.importances_mean, index=FEATURES).sort_values().tail(15)
    ax.barh(s.index, s.values, color=["#f2b134" if f in SEASONAL_FEATURES else "#4c9be8" for f in s.index])
    ax.set(title=f"{tname}: top 15 features", xlabel="Increase in MAE when shuffled (MW)")
    seasonal_share = imp.importances_mean[[f in SEASONAL_FEATURES for f in FEATURES]].clip(0).sum() / \
                     imp.importances_mean.clip(0).sum()
    print(f"{tname}: seasonal features account for {seasonal_share:.0%} of total importance")
fig.suptitle("Permutation importance (yellow = seasonal features)")
plt.tight_layout(); save_fig("07_feature_importance.png"); plt.show()

## 15. Summary, limitations and ethics
**What this section does:** prints the key results for the report, then lists limitations and ethical points for the team to discuss and expand.

**Why:** the report needs to explain not only how accurate the model is, but where it should and should not be trusted. These points support the Results, Issues Faced, Potential for Wider Adoption and Conclusions sections.

In [ ]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"Daily minimum fell between 10:00 and 15:00 on {midday_share:.0%} of days")
print(f"Training: {len(train)} days | Hold-out: {len(hold)} days ({HOLDOUT_START} to {hold.index.max():%Y-%m-%d})\n")
for t, tname in TARGETS.items():
    cv_best = cv_table[(cv_table["Target"] == tname) & (cv_table["Model"] == best_reg[t])].iloc[0]
    cv_seas = cv_table[(cv_table["Target"] == tname) & (cv_table["Model"] == "Baseline: seasonal average")].iloc[0]
    ho = hold_table[(hold_table["Target"] == tname) & (hold_table["Model"] == best_reg[t])].iloc[0]
    ho_base = hold_table[(hold_table["Target"] == tname) & hold_table["Model"].str.startswith("Baseline")]["MAE (MW)"].min()
    cl = clf_table[(clf_table["Target"] == tname) & (clf_table["Model"] == best_clf[t])].iloc[0]
    print(f"{tname}")
    print(f"  Regression: {best_reg[t]}")
    print(f"    CV MAE {cv_best['MAE (MW)']:.0f} MW vs seasonal-average baseline {cv_seas['MAE (MW)']:.0f} MW "
          f"({1 - cv_best['MAE (MW)'] / cv_seas['MAE (MW)']:.0%} better)")
    print(f"    Hold-out MAE {ho['MAE (MW)']:.0f} MW, MAPE {ho['MAPE (%)']:.1f}% "
          f"({1 - ho['MAE (MW)'] / ho_base:.0%} better than best baseline)")
    print(f"  Risk flag (below {THRESHOLDS[t]:,.0f} MW): {best_clf[t]}")
    print(f"    CV: caught {cl['Caught']} of {cl['Risky days']} risky days (recall {cl['Recall']:.0%}), "
          f"{cl['False alarms']} false alarms (precision {cl['Precision']:.0%})\n")
print(f"Figures and tables saved in '{RESULTS_DIR}/': {sorted(os.listdir(RESULTS_DIR))}")

### Limitations
- **No solar or cloud data.** Midday minimum load is driven mainly by how much rooftop solar is generating, which depends on cloud cover. The dataset has only temperature and wind, so the model cannot tell a sunny mild day from a cloudy mild day. Adding solar irradiance or cloud cover (for example, from the Bureau of Meteorology) is the most valuable next step.
- **Observed rather than forecast weather.** The model treats actual temperature as a perfect forecast, so real-world accuracy would be lower.
- **Only two years of data.** Each season appears just once or twice, and rare events (such as a record-low spring holiday) appear only a few times.
- **Hold-out season.** April to June contains few risky days, so flagging performance is judged mainly by cross-validation (Sections 11 and 12).
- **A moving target.** Rooftop solar keeps growing, so minimum load keeps falling year on year. A fixed "lowest 10%" threshold will be crossed more often over time, and the model needs regular retraining, similar to fine-tuning a pre-trained model on recent data (Week 8 lab).
- **One weather station.** Sydney Airport represents all of NSW, but rooftop solar is spread across the state.

### Ethical and AI-for-Good considerations
- **Enabling renewables.** Reliable minimum-load forecasts help operators keep the grid stable without unnecessarily limiting rooftop solar, so more clean energy can be used.
- **Asymmetric risk.** Predicting a minimum that is too high (missing a risky day) is more dangerous than a false alarm, because operators would be unprepared. This is why recall is emphasised and seasonal bias is checked.
- **Fairness to solar owners.** If forecasts trigger curtailment of rooftop solar, households lose generation they paid for. Over-cautious forecasts would shift cost onto them, so accuracy has a direct equity dimension.
- **Human control and transparency** (Week 6 framework). The forecast should support operators, not act automatically. Feature importance and the flag charts help operators judge when to trust it.
- **Privacy.** The data is aggregated at state level, so individual privacy risk is very low. Extending the model to household smart-meter data would raise real privacy concerns.